## Imports and paths

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)

In [2]:
# File path
PROJECT_DIR = Path.cwd()
UNIFORM_DIR = PROJECT_DIR / "Cleaned Data" / "Uniform column format"


# Output path
OUTPUT_DIR = PROJECT_DIR / "Cleaned Data" / "002 Data Clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UNENCODED_OUTPUT_DIR = OUTPUT_DIR / "cleaned_unencoded"
ENCODED_OUTPUT_DIR = OUTPUT_DIR / "cleaned_encoded"
UNENCODED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ENCODED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# 6 Countries
COUNTRIES = {"AUS": "Australia",  "BRA": "Brazil",  "CAN": "Canada",  "GBR": "United Kingdom",  "IND": "India",  "USA": "United States"}

# The date on which the <14-day> rolling average for H6M_Facial Coverings first reaches <2.5> or higher will be defined as the mandate date.
# If the threshold is not reached, the first date reaching the maximum rolling average will be used.
ROLLING_DAYS = 14
MANDATE_THRESHOLD = 2.5

## 1 Helper functions

### 1.1 Helper functions for cleaning data

In [4]:
# Read function
def read_csv_clean(path):
    return pd.read_csv(path, na_values=[" ", "__NA__", ""], keep_default_na=True, low_memory=False)         # Treat NA as missing values


# Parsing Time
def parse_endtime(x):
    return pd.to_datetime(x, dayfirst=True, errors="coerce")


# Count the sample size at before/after mandate
def count_mandate_period_rows(df, period_col="within_mandate_period"):
    counts = df[period_col].value_counts(dropna=False)
    return {"Before_mandate_rows": int(counts.get(0, 0)),"After_mandate_rows": int(counts.get(1, 0)),"Unknown_mandate_rows": int(df[period_col].isna().sum())}


# Return columns treated as categorical variables
def get_categorical_columns(df):
    return df.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()


# Imputing missing values with "N/A"
def fill_categorical_missing_with_NA(df):
    out = df.copy()
    categorical_cols = get_categorical_columns(out)

    if len(categorical_cols) > 0:
        out[categorical_cols] = out[categorical_cols].fillna("N/A")
    return out


# Sort "i12_health_xxx"
def sort_i12_columns(cols):
    def extract_number(col):
        match = re.search(r"i12_health_(\d+)", col)
        if match:
            return int(match.group(1))
        return 9999
    return sorted(cols, key=extract_number)

### 1.2 Helper functions for recoding

In [5]:
# Recode numbers of family members
def recode_household(x):                                                                                # Recode household
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x in [str(i) for i in range(1, 8)]:                                                              # If 0<x<8, then to x
        return int(x)
    if x == "8 or more":                                                                                # If >= 8, then to 8
        return 8
    if x in ["Prefer not to say", "Prefer not to answer", "Don't know", "Dont know","N/A","NA"]:        # If else, then to NA
        return np.nan
    return np.nan


# Recode risk perception
def recode_agreement_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    agreement_map = {                                                                                   # Some Formatting Issues
        "7 - Agree": 7,
        "7 – Agree": 7,
        "7 — Agree": 7,
        "7": 7,
        "6": 6,
        "5": 5,
        "4": 4,
        "3": 3,
        "2": 2,
        "1 - Disagree": 1,
        "1 – Disagree": 1,
        "1 — Disagree": 1,
        "1": 1}
    return agreement_map.get(x, np.nan)


# Recode frequency of protective behaviors
def recode_frequency_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    frequency_map = {"Always": 5, "Frequently": 4, "Sometimes": 3, "Rarely": 2, "Not at all": 1}        # Definition scale
    return frequency_map.get(x, np.nan)


# Convert some variable to numerical form
def recode_numeric_later_columns(df):
    out = df.copy()
    if "household_size" in out.columns:
        out["household_size"] = out["household_size"].apply(recode_household)
    for col in ["r1_1", "r1_2","r1_6","r1_7"]:
        if col in out.columns:
            out[col] = out[col].apply(recode_agreement_value)
    for col in COMMON_I12_OUTCOME_COLS:
        if col in out.columns:
            out[col] = out[col].apply(recode_frequency_value)
    return out

# One-hot encode categorical variables
def encode_categorical_variables(df):
    out = df.copy()                                                                                                                 # Copy
    target_cols = ["face_mask_behaviour_binary","non_mask_protective_behaviour_binary","overall_protective_behaviour_binary"]       # Exclude target columns
    exclude_from_encoding = set(target_cols)
    categorical_cols = [col for col in get_categorical_columns(out)if col not in exclude_from_encoding]
    if len(categorical_cols) > 0:
        out = pd.get_dummies(out,columns=categorical_cols,drop_first=True,dtype=bool)
    return out

### 1.3 Helper functions for mandate period

In [6]:
# Compute mandate date
def compute_mandate_dates(oxcgrt_df):

    df = oxcgrt_df.copy()                                                                           # Copy
    df["policy_date"] = pd.to_datetime( df["Date"].astype(str), format="%Y%m%d", errors="coerce")   # Convert date
    df["H6M_Facial Coverings"] = pd.to_numeric( df["H6M_Facial Coverings"],errors="coerce")         # Convert H6M_Facial Coverings
    df = df.dropna(subset=["policy_date", "H6M_Facial Coverings"]).copy()                           # Copy
    df = df.sort_values("policy_date")                                                              # Sort


    # 1 Region mandate dates
    region_rows = []
    region_df = df[df["RegionName"].notna()].copy()

    for region_name, g in region_df.groupby("RegionName"):
        g = g.sort_values("policy_date").copy()
        g["rolling_h6"] = (g["H6M_Facial Coverings"].rolling(ROLLING_DAYS, min_periods=ROLLING_DAYS).mean())    # 14-day rolling average to identify “ongoing” mask policy phases
        valid_rolling = g[g["rolling_h6"].notna()].copy()                                                        # Keep valid rolling average values

        if len(valid_rolling) == 0:
            continue

        hit = valid_rolling[valid_rolling["rolling_h6"] >= MANDATE_THRESHOLD]                                    # 14-day threshold

        if len(hit) > 0:
            mandate_start_date = hit["policy_date"].iloc[0]                                                      # First date reaching the threshold
            mandate_rule = "threshold"
        else:
            max_idx = valid_rolling["rolling_h6"].idxmax()                                                       # First date reaching the maximum rolling average
            mandate_start_date = valid_rolling.loc[max_idx, "policy_date"]
            mandate_rule = "max_rolling"

        region_rows.append({"state": region_name, "mandate_start_date": mandate_start_date, "mandate_rule": mandate_rule})

    region_mandates = pd.DataFrame(region_rows, columns=["state", "mandate_start_date", "mandate_rule"])
    return region_mandates


# Audit table for store information
def build_region_mandate_audit(yougov_df, region_mandates, country_code, country_name):

    states = (yougov_df["state"].dropna().astype(str).sort_values().unique().tolist())              # Extract the regions that actually appear in the YouGov data
    audit = pd.DataFrame({"state": states})
    audit = audit.merge(region_mandates,on="state",how="left")                                      # Merge region mandates

    audit["country_code"] = country_code
    audit["country"] = country_name
    audit = audit[["country_code","country","state","mandate_start_date","mandate_rule"]]       # View mandate dates by region
    return audit


# Add the mandate period (0/1)
def add_mandate_period(yougov_df,region_mandates):

    out = yougov_df.copy()                                                                                                          # Copy
    out["endtime"] = parse_endtime(out["endtime"])                                                                                  # Analyze the survey completion date
    out = out.merge(region_mandates,on="state",how="left")                                                                          # Merge

    out["within_mandate_period"] = np.nan
    valid = out["endtime"].notna() & out["mandate_start_date"].notna()
    out.loc[valid, "within_mandate_period"] = (out.loc[valid, "endtime"] >= out.loc[valid, "mandate_start_date"]).astype(int)       # Categories
    out = out.drop(columns=["mandate_start_date", "mandate_rule"],errors="ignore")                                                 # Delete unnecessary columns
    return out

### 1.4 Helper functions for behaviour outcomes

In [7]:
# Convert behaviour scale to binary compliance outcome
def binary_from_scale(series, threshold=4):                                             # Protective behaviors frequency is (1~5) (1 = Not at all   2 = Rarely   3 = Sometimes   4 = Frequently   5 = Always) 
    return np.where(series.notna(),np.where(series >= threshold, 1, 0),np.nan)          # if Frequency >= 4 → 1 ; if Frequency < 4 → 0


# Create face mask, non-mask and overall behaviour outcomes
def add_behaviour_outcomes(df):
    out = df.copy()                                                                                                             # Copy
    out["face_mask_behaviour_scale"] = out[FACE_MASK_OUTCOME_COLS].median(axis=1, skipna=False)                                 # Face mask behaviour scale
    out["face_mask_behaviour_binary"] = binary_from_scale(out["face_mask_behaviour_scale"])                                     # Face mask behaviour binary

    out["non_mask_protective_behaviour_scale"] = out[NON_MASK_PROTECTIVE_BEHAVIOUR_OUTCOME_COLS].median(axis=1, skipna=False)   # No mask protective behaviour scale
    out["non_mask_protective_behaviour_binary"] = binary_from_scale(out["non_mask_protective_behaviour_scale"])                 # No mask protective behaviour binary

    out["overall_protective_behaviour_scale"] = out[OVERALL_PROTECTIVE_BEHAVIOUR_OUTCOME_COLS].median(axis=1, skipna=False)     # Overall protective behaviour scale
    out["overall_protective_behaviour_binary"] = binary_from_scale(out["overall_protective_behaviour_scale"])                   # Overall protective behaviour binary

    return out

### 1.5 Helper functions for cleaning data (Delete the unnecessary column)

In [8]:
# Drop i12 columns
def drop_non_common_i12_columns(df):
    out = df.copy()                                                                         # Copy
    i12_cols = [col for col in out.columns if col.startswith("i12_health_")]                # If not used in the common cross-national outcome definition
    non_common_i12_cols = [col for col in i12_cols if col not in COMMON_I12_OUTCOME_COLS]   # Then delete
    if len(non_common_i12_cols) > 0:
        out = out.drop(columns=non_common_i12_cols)
    return out


# Drop rows with missing values in numeric or datetime columns
def drop_rows_with_numeric_or_datetime_missing(df):                                                             # Why aren't missing values in categorical variables removed? ： If they are deleted, it will result in an extremely large number of missing samples, rendering the data statistically insignificant. These can be replaced with “NA” and encoded as a separate column, but this cannot be done with numeric columns.
    out = df.copy()                                                                                             # Copy
    numeric_or_datetime_cols = out.select_dtypes(include=["number", "datetime64[ns]"]).columns.tolist()         # "numeric_or_datetime_cols" is the name of the column checked in this round
    rows_before = out.shape[0]
    if len(numeric_or_datetime_cols) > 0:
        out = out.dropna(subset=numeric_or_datetime_cols).copy()                                                # "out" is the data after removed missing values 
    rows_after = out.shape[0]                                                                                   
    rows_dropped = rows_before - rows_after                                                                     # "rows_dropped" indicates the number of rows deleted due to missing numeric or date values
    return out, rows_dropped, numeric_or_datetime_cols                                                          


# Collapse "d1_health_xxx" variables into one comorbidity variable
def collapse_comorbidity_variables(df):
    out = df.copy()                                                                                             # Copy
    d1_cols = [col for col in out.columns if col.startswith("d1_health_")]                                      # Find all variables
    if len(d1_cols) == 0:
        return out
    out["d1_comorbidities"] = "N/A"                                                                             # Set default
    disease_cols = [col for col in d1_cols if col not in ["d1_health_98", "d1_health_99"]]                      # Exclude two columns, as they typically do not refer to specific diseases
    if len(disease_cols) > 0:
        has_comorbidity = (out[disease_cols].astype(str).eq("Yes").any(axis=1))                                 # If "Yes", then to Yes
        out.loc[has_comorbidity,"d1_comorbidities"] = "Yes"
    if "d1_health_99" in out.columns:
        out.loc[out["d1_health_99"].astype(str).eq("Yes"),"d1_comorbidities"] = "No"                            # If "No", then to no
    if "d1_health_98" in out.columns:
        out.loc[out["d1_health_98"].astype(str).eq("Yes"),"d1_comorbidities"] = "Prefer_not_to_say"             # If unwilling to answer questions about their health, then to "Prefer_not_to_say"
    out = out.drop(columns=d1_cols)
    return out


# Create time variable
def add_week_number(df):
    out = df.copy()
    start_date = out["endtime"].min()                                                                           # Grouped into 14-day intervals
    out["week_number"] = (((out["endtime"] - start_date).dt.days // 14) + 1)                                    # Its relative Time
    return out


# Drop raw "i12_health_xxx" items after outcome construction
def drop_original_i12_columns(df):
    out = df.copy()
    i12_cols = [col for col in out.columns if col.startswith("i12_health_")]                                    # Once the outcome has been created
    if len(i12_cols) > 0:                                                                                       # delete the original i12_health_xxx behavior items
        out = out.drop(columns=i12_cols)
    return out


# Save function
def save_csv_with_true_false(df, path):
    export_df = df.copy()
    bool_cols = export_df.select_dtypes(include=["bool"]).columns.tolist()
    for col in bool_cols:
        export_df[col] = export_df[col].map({True: "TRUE",False: "FALSE"})
    export_df.to_csv(path,index=False,encoding="utf-8-sig")

## 2. Main preprocessing

### 2.1 Read data

In [9]:
# Create 3 dictionaries
raw_yougov_data = {}                        # Yougov data
raw_oxcgrt_data = {}                        # OxCGRT data
retained_columns_by_country = {}            # Variable names retained for each country after screening for missing values

# Read
for code, country_name in COUNTRIES.items():

    yougov_path = UNIFORM_DIR / f"YouGov_{country_name}.csv"
    oxcgrt_path = UNIFORM_DIR / f"OxCGRT_{country_name}.csv"

    yougov_df = read_csv_clean(yougov_path)
    oxcgrt_df = read_csv_clean(oxcgrt_path)

    raw_yougov_data[code] = yougov_df.copy()
    raw_oxcgrt_data[code] = oxcgrt_df.copy()

    missing_rate = yougov_df.isna().mean()                                                                                          # Filter Variablesand
    retained_cols = missing_rate[missing_rate <= 0.3].index.tolist()                                                                # Retain variables (attrition rate <= 30%)
    retained_cols = [col for col in retained_cols if col not in ["cantril_ladder","qweek","weight","household_children"]]           # If the 4 columns are retained, this will result in a large number of missing values in the sample
    retained_columns_by_country[code] = retained_cols                                                                               # Save

### 2.2  Identify outcome (Face mask behaviour and Overall protective behaviour)

In [10]:
# Common columns retained in all countries
common_columns = sorted(set.intersection(*[set(cols) for cols in retained_columns_by_country.values()]))
# "i12_health_xxx" Common columns retained in all countries
common_i12_cols = sort_i12_columns([col for col in common_columns if col.startswith("i12_health_")])


# Face mask outcome
FACE_MASK_OUTCOME_COLS = ["i12_health_1"]                                                                                       # Face mask outcome
missing_face_mask_cols = [col for col in FACE_MASK_OUTCOME_COLS if col not in common_i12_cols]
if len(missing_face_mask_cols) > 0:
    raise ValueError(f"These face mask outcome columns are not common across all countries: {missing_face_mask_cols}")


# Cross-national outcome item sets
COMMON_I12_OUTCOME_COLS = common_i12_cols                                                           
NON_MASK_PROTECTIVE_BEHAVIOUR_OUTCOME_COLS = [col for col in COMMON_I12_OUTCOME_COLS if col not in FACE_MASK_OUTCOME_COLS]      # No mask protective behaviour
OVERALL_PROTECTIVE_BEHAVIOUR_OUTCOME_COLS = ( COMMON_I12_OUTCOME_COLS.copy() )                                                  # Overall protective behaviour


# Display
outcome_definition_table = pd.DataFrame({
    "outcome": ["face_mask_behaviour_binary","non_mask_protective_behaviour_binary","overall_protective_behaviour_binary"],
    "columns_used": [", ".join(FACE_MASK_OUTCOME_COLS),", ".join(NON_MASK_PROTECTIVE_BEHAVIOUR_OUTCOME_COLS),", ".join(OVERALL_PROTECTIVE_BEHAVIOUR_OUTCOME_COLS)],
    "number_of_columns": [len(FACE_MASK_OUTCOME_COLS),len(NON_MASK_PROTECTIVE_BEHAVIOUR_OUTCOME_COLS),len(OVERALL_PROTECTIVE_BEHAVIOUR_OUTCOME_COLS)]})
display(outcome_definition_table)

,outcome,columns_used,number_of_columns
0,face_mask_behaviour_binary,i12_health_1,1
1,non_mask_protective_behaviour_binary,"i12_health_2, i12_health_3, i12_health_4, i12_...",13
2,overall_protective_behaviour_binary,"i12_health_1, i12_health_2, i12_health_3, i12_...",14


### 2.3 Clean data

In [11]:
yougov_cleaned_unencoded = {}                                                                           # Store each country's cleaned but uncoded data
yougov_cleaned_encoded = {}                                                                             # Store each country's cleaned encoded data
summary_rows = []                                                                                       # Record any changes during the cleaning process in each country
mandate_audit_tables = []                                                                               # Mandate date table

for code, country_name in COUNTRIES.items():

    # 1 Start from country-specific retained columns
    df = raw_yougov_data[code][retained_columns_by_country[code]].copy()
    oxcgrt_df = raw_oxcgrt_data[code].copy()

    # 2 Remove non-common i12_health_* columns
    df = drop_non_common_i12_columns(df)

    # 3 Compute mandate dates
    region_mandates = compute_mandate_dates(oxcgrt_df)
    mandate_audit = build_region_mandate_audit(yougov_df=df,region_mandates=region_mandates,country_code=code,country_name=country_name)
    mandate_audit_tables.append(mandate_audit)

    # 4 Add mandate-period indicator
    df = add_mandate_period(yougov_df=df, region_mandates=region_mandates)
    counts_before_numeric_drop = count_mandate_period_rows(df)

    # 5 Recode variables that should be numeric
    df = recode_numeric_later_columns(df)

    # 6 Add behaviour outcomes
    df = add_behaviour_outcomes(df)

    # 7 Drop rows with missing values in numeric or datetime columns
    rows_before_numeric_drop = df.shape[0]
    df, rows_dropped_numeric_or_datetime, numeric_or_datetime_cols_checked = (drop_rows_with_numeric_or_datetime_missing(df))
    rows_after_numeric_drop = df.shape[0]
    counts_after_numeric_drop = count_mandate_period_rows(df)


    df["within_mandate_period"] = df["within_mandate_period"].astype(int)                   # Convert mandate-period indicator to integer
    df = fill_categorical_missing_with_NA(df)                                               # Fill remaining categorical missing values with N/A
    df = collapse_comorbidity_variables(df)                                                 # Collapse comorbidity variables
    df = add_week_number(df)                                                                # Add week time variable
    df = drop_original_i12_columns(df)                                                      # Drop raw "i12_health_xxx" columns after outcome construction


    # Save (Unencoded cleaned data)
    safe_country_name = country_name.replace(" ", "_")
    unencoded_path = (UNENCODED_OUTPUT_DIR / f"YouGov_{safe_country_name}_cleaned_unencoded.csv")
    df.to_csv(unencoded_path,index=False,encoding="utf-8-sig")


    yougov_cleaned_unencoded[code] = df.copy()

    
    # Save (Encoded cleaned data)
    encoded_df = encode_categorical_variables(df)
    encoded_path = (ENCODED_OUTPUT_DIR / f"YouGov_{safe_country_name}_cleaned_encoded.csv")
    save_csv_with_true_false( df=encoded_df,path=encoded_path)
    yougov_cleaned_encoded[code] = encoded_df.copy()


    # Store summary information for display only
    summary_rows.append({
        "country": country_name,

        "before_mandate_before_numeric_missing_drop": counts_before_numeric_drop["Before_mandate_rows"],
        "after_mandate_before_numeric_missing_drop": counts_before_numeric_drop["After_mandate_rows"],
        "unknown_mandate_before_numeric_missing_drop": counts_before_numeric_drop["Unknown_mandate_rows"],

        "before_mandate_after_numeric_missing_drop": counts_after_numeric_drop["Before_mandate_rows"],
        "after_mandate_after_numeric_missing_drop": counts_after_numeric_drop["After_mandate_rows"],
        "unknown_mandate_after_numeric_missing_drop": counts_after_numeric_drop["Unknown_mandate_rows"],

        "rows_before_numeric_missing_drop": rows_before_numeric_drop,
        "rows_after_numeric_missing_drop": rows_after_numeric_drop,
        "rows_dropped_due_to_numeric_or_datetime_missing": rows_dropped_numeric_or_datetime,

        "unencoded_columns": df.shape[1],
        "encoded_columns": encoded_df.shape[1]})
        

## 3 Visualization of Processing Results

### 3.1 Mandate date table

In [12]:
region_mandate_dates = pd.concat(mandate_audit_tables,ignore_index=True)
region_mandate_dates.to_csv(OUTPUT_DIR / "region_mandate_dates.csv",index=False,encoding="utf-8-sig")
mandate_dates_preview = pd.concat([region_mandate_dates.head(5),region_mandate_dates.tail(5)])
display(mandate_dates_preview)

,country_code,country,state,mandate_start_date,mandate_rule
0,AUS,Australia,Australian Capital Territory,2021-07-04,threshold
1,AUS,Australia,New South Wales,2021-07-06,threshold
2,AUS,Australia,Northern Territory,2021-06-30,threshold
3,AUS,Australia,Queensland,2021-01-14,threshold
4,AUS,Australia,South Australia,2021-07-23,threshold
133,USA,United States,Washington,2020-07-06,threshold
134,USA,United States,Washington DC,2020-05-23,threshold
135,USA,United States,West Virginia,2020-11-19,threshold
136,USA,United States,Wisconsin,2020-07-22,threshold
137,USA,United States,Wyoming,2020-05-14,max_rolling


### 3.2 Unencoded date

In [13]:
unencoded_mandate_summary = []
for code, country_name in COUNTRIES.items():
    df = yougov_cleaned_unencoded[code]
    unencoded_mandate_summary.append({"country": country_name,"before_mandate_rows": int((df["within_mandate_period"] == 0).sum()),"after_mandate_rows": int((df["within_mandate_period"] == 1).sum()),"columns": df.shape[1],})
unencoded_mandate_summary = pd.DataFrame(unencoded_mandate_summary)
display(unencoded_mandate_summary)

,country,before_mandate_rows,after_mandate_rows,columns
0,Australia,15757,25988,27
1,Brazil,1479,8639,39
2,Canada,15856,24055,26
3,United Kingdom,1689,30729,35
4,India,1736,13629,37
5,United States,1796,15419,28


### 3.3 Encoded date

In [14]:
encoded_mandate_summary = []
for code, country_name in COUNTRIES.items():
    df = yougov_cleaned_encoded[code]
    encoded_mandate_summary.append({"country": country_name,"before_mandate_rows": int((df["within_mandate_period"] == 0).sum()), "after_mandate_rows": int((df["within_mandate_period"] == 1).sum()),"columns": df.shape[1],})
encoded_mandate_summary = pd.DataFrame(encoded_mandate_summary)
display(encoded_mandate_summary)

,country,before_mandate_rows,after_mandate_rows,columns
0,Australia,15757,25988,67
1,Brazil,1479,8639,130
2,Canada,15856,24055,71
3,United Kingdom,1689,30729,81
4,India,1736,13629,127
5,United States,1796,15419,102


### 3.4 Summary table

In [15]:
summary_table = pd.DataFrame(summary_rows)
display(summary_table)

,country,before_mandate_before_numeric_missing_drop,after_mandate_before_numeric_missing_drop,unknown_mandate_before_numeric_missing_drop,before_mandate_after_numeric_missing_drop,after_mandate_after_numeric_missing_drop,unknown_mandate_after_numeric_missing_drop,rows_before_numeric_missing_drop,rows_after_numeric_missing_drop,rows_dropped_due_to_numeric_or_datetime_missing,unencoded_columns,encoded_columns
0,Australia,25155,28678,0,15757,25988,0,53833,41745,12088,27,67
1,Brazil,1497,8811,0,1479,8639,0,10308,10118,190,39,130
2,Canada,21429,26943,0,15856,24055,0,48372,39911,8461,26,71
3,United Kingdom,14624,49908,0,1689,30729,0,64532,32418,32114,35,81
4,India,1800,14345,0,1736,13629,0,16145,15365,780,37,127
5,United States,9529,24411,0,1796,15419,0,33940,17215,16725,28,102
